# Phase 14 — Streamlit UI (headless smoke test)

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** drive the exact graph the UI calls, headlessly, before launching Streamlit.

**Note:** demo single-stock analysis on CLT-005 (holds NVDA), not on fund-only clients.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env', override=True)  # override=True: beat VS Code's stale cached env vars

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

## 1. Stream graph events the way the UI does

In [ ]:
import asyncio
from app.graph.builder import GraphBuilder
graph = GraphBuilder().with_all().build()
cfg = {'configurable': {'thread_id': 'CLT-005-nb14'}}

async def drive():
    inp = {'messages': [('user', 'technical analysis on my NVDA position')],
           'client_id': 'CLT-005', 'session_id': 'nb-14'}
    async for ev in graph.astream_events(inp, cfg, version='v2'):
        if ev['event'] == 'on_chain_start':
            print('->', ev['name'])

await drive()

## 2. Launch the real UI (from a terminal, not the notebook)

In [ ]:
print('Run in a terminal:  make ui   # streamlit run ui/streamlit_app.py')

## ✅ Acceptance check

In [ ]:
print('If agent events streamed above for CLT-005, the UI path is wired. Phase 14: PASS')